# R_P00 -- Prepare the panel (data preparation, R)

Reads every export under ROOT (default D:/LKT/RWDR/data; the old D:/LKT/TST_ArtalR until MIGRATE_DATA.bat has moved the data), builds the pixel panel with the SAME rules as the Python P00, averages the benchmark (BM) sites of each sub-watershed, codes every row for the fragment rule and reads your fund workbook. v20.57: it sets NO design -- each model notebook sets and applies its own (the options set to "data" are chosen FROM THE DATA, never from the effect).

In [ ]:
# the bundle's R folder (works from rstudio/, jupyter/ or the project root)
R_HOME_DIR <- normalizePath(if (file.exists("lib/reward_paths.R")) "." else "..", winslash = "/")
for (f in c("reward_paths.R", "reward_design.R", "reward_prep.R", "reward_models_core.R")) source(file.path(R_HOME_DIR, "lib", f), local = environment())
cat("data root:", ROOT, "| output:", OUTPUT_DIR, "\n")
invisible(confirm_packages())   # v20.55: are ALL R packages installed? (pre-built first; a missing one is installed now when AUTO_INSTALL_PACKAGES)

**The design is NOT set here (v20.57).** R_P00 builds the panel ONCE with every row -- all rings, years and seasons, every sub-watershed and every fragment of another sub-watershed, each row coded (`fragment`) -- and never has to be re-run for a design choice. The timing (your fund workbook by default), the dose, the control rings, the years, the seasons, OVERLAP_ROWS, FRAGMENT_RULE and POOLED_FE are set in the settings cell of EACH MODEL notebook (R_M01 ... R_M45, R_RUN_ALL_MODELS, R_D01) and applied when that model runs. The defaults live in `lib/reward_paths.R`. Change `ROOT` there if your exports are elsewhere.

In [ ]:
# v20.57: nothing to set here for the design -- it is set in each MODEL notebook (defaults: lib/reward_paths.R).
# v20.59 (YOUR RULE): PERIOD_RULE (lib/reward_paths.R) sets the panel's post / pre -- "treat" = the exports' Treat column (1 = post, 0 = pre),
# "year" = Year >= TREATMENT_YEAR, "both" = the two must agree (a disagreeing row leaves). buff_km 0 = the treatment area, 1-5 = the control
# rings. R_P00 confirms both on every input file -> input_design_audit_R.csv.
# The panel-building rules (pixel linkage, duplicates, masked zeros, negative covariates) stay in lib/reward_paths.R / reward_prep.R.
cat("design defaults (a model notebook overrides each):", DESIGN_MODE, "| timing", TREATMENT_TIMING, "| period rule (R_P00)", PERIOD_RULE, "| rings", paste(CONTROL_RINGS, collapse = ","),
    "| seasons", paste(SEASONS, collapse = ","), "| fragments", FRAGMENT_RULE, "| overlap", OVERLAP_ROWS, "| pooled FE", POOLED_FE, "\n")


**Build the panel** -- exports, harmonised columns, masked zeros as missing, the sub-watershed from the shapefile (confirmed or corrected), every row coded for the fragment rule per export file, shifted grids linked, repeated rows dropped whole (the newer export's row kept as it is -- v20.58; DEDUP_FILL_FROM_DUPLICATES in lib/reward_prep.R), the BM means; the fund workbook's timing and dose tables (results/FUND) for you to read -- the models apply them when they run.

In [ ]:
panel <- run_prep()

**The design the DEFAULTS give** (export breaks, fill years, spillover into inner rings; the fund timing per sub-watershed) -- what a model uses when its settings are left as in lib/reward_paths.R. Each model applies its own settings when it runs.

In [ ]:
design <- prepare_design(); str(design[setdiff(names(design), c("choices", "notes"))])


**The outcome screen** -- which outcome-years are fill values (not data), per outcome.

In [ ]:
scr <- outcome_screen_R(OUTCOMES, design)   # v20.58: every outcome's usable years -- all rows at once, or out of core beyond 98 % of the RAM
fwrite(scr, file.path(RESULTS_DIR, "OUTCOME_SCREEN_R.csv")); print(scr)